# Week 20, Day 3: Estimating information from spikes

Companion notebook for the Engram lesson at https://engram-neuroai.vercel.app/learn/w020/d3.
Run the cells in order. Exercises have a checks cell after them: run it to test your answer.

### Exercise: Information from nothing

Write `plugin_mi(s, r, S, R)` that builds the joint frequency table from integer arrays of stimuli and responses and returns its mutual information. For 500 simulated experiments with 8 stimuli × 20 trials and responses drawn uniformly from 16 bins, independently of the stimulus, store the estimates in `naive` and compare their mean with the formula.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
rng = np.random.default_rng(1998)

def entropy(p):
    p = np.asarray(p, float); p = p[p > 0]
    return -np.sum(p * np.log2(p))

def plugin_mi(s, r, S, R):
    J = np.zeros((S, R))
    np.add.at(J, (s, r), 1)
    J /= J.sum()
    return ...                                     # H(S) + H(R) - H(S, R)

S, R, n_per = 8, 16, 20
s = np.repeat(np.arange(S), n_per)
naive = np.array([plugin_mi(s, rng.integers(0, R, len(s)), S, R) for _ in range(500)])
print(f"plug-in information of unrelated data: {naive.mean():.3f} bits (formula {7 * 15 / (2 * 160 * np.log(2)):.3f})")

In [ ]:
import numpy as np
def test_positive():
    "the plug-in information of independent data is clearly positive on average (above 0.2 bits)"
    assert np.mean(naive) > 0.2

def test_formula():
    "the average bias is within 25% of (S - 1)(R - 1) / (2 N ln 2)"
    pred = 7 * 15 / (2 * 160 * np.log(2))
    assert abs(np.mean(naive) - pred) / pred < 0.25

test_positive()
test_formula()
print("All checks passed")

### Exercise: Shuffle correction on a tuned neuron

A tuned neuron's count (capped at 15) depends on 8 orientations. For 10, 40 and 160 trials per stimulus, compute the naive plug-in information (`naive_est[n]`) and the shuffle-corrected value (`corrected[n]`, naive minus the mean of 100 shuffles). Compare with `true_info` computed from the exact joint table.

In [ ]:
from math import lgamma
rates = 2 + 8 * np.exp(2 * (np.cos(np.deg2rad(2 * (np.arange(8) * 22.5 - 90))) - 1))
kk = np.arange(16)
P = np.exp(kk[None] * np.log(rates[:, None]) - rates[:, None] - np.array([lgamma(k + 1) for k in kk])[None])
P[:, -1] += 1 - P.sum(axis=1)                       # lump counts of 15 or more into the last bin
J_true = P / 8
true_info = entropy(J_true.sum(1)) + entropy(J_true.sum(0)) - entropy(J_true.ravel())

r_c = np.random.default_rng(160)
naive_est, corrected = {}, {}
for n_per in (10, 40, 160):
    s = np.repeat(np.arange(8), n_per)
    r = np.minimum(r_c.poisson(rates[s]), 15)
    naive_est[n_per] = plugin_mi(s, r, 8, 16)
    shuffled = [plugin_mi(r_c.permutation(s), r, 8, 16) for _ in range(100)]
    corrected[n_per] = ...
print(f"true information {true_info:.3f} bits")
for n_per in naive_est: print(f"{n_per:4d} trials per stimulus: naive {naive_est[n_per]:.3f}, shuffle-corrected {corrected[n_per]:.3f}")

In [ ]:
import numpy as np
def test_naive_high():
    "with 10 trials per stimulus, the naive estimate overshoots the true information by more than 0.15 bits"
    assert naive_est[10] - true_info > 0.15

def test_corrected():
    "the naive estimate overshoots at every sample size; with 40 trials the correction brings it closer to the truth, and with 160 both are within 0.06 bits"
    assert all(naive_est[n] > true_info for n in naive_est)
    assert abs(corrected[40] - true_info) < abs(naive_est[40] - true_info) and abs(corrected[160] - true_info) < 0.06

test_naive_high()
test_corrected()
print("All checks passed")

### Exercise: Precise and jittered spike timing

A neuron fires at stimulus-locked times (a fixed pattern of 400 events in 4 s, 2 ms bins), on 200 repeats, missing some events and adding a few random spikes. The precise neuron's spikes keep their times; the jittered neuron's are moved by about ±3 bins. Write `direct_info(trains, L)` returning total entropy, noise entropy and information (bits per word), and compare the two neurons with words of 4 bins.

In [ ]:
r_d = np.random.default_rng(1997)
T_bins, reps, L = 2000, 200, 4
events = np.sort(r_d.choice(T_bins, 400, replace=False))       # times the stimulus drives a spike

def make_trains(jitter):
    trains = np.zeros((reps, T_bins), int)
    for k in range(reps):
        keep = events[r_d.random(len(events)) < 0.8]
        t = np.clip(keep + np.round(jitter * r_d.normal(size=len(keep))).astype(int), 0, T_bins - 1)
        trains[k, t] = 1
        trains[k, r_d.random(T_bins) < 0.01] = 1               # a few spontaneous spikes
    return trains

def direct_info(trains, L):
    n_words = trains.shape[1] // L
    words = trains[:, :n_words * L].reshape(trains.shape[0], n_words, L) @ (2 ** np.arange(L))   # repeats x times
    H_total = entropy(np.bincount(words.ravel(), minlength=2 ** L) / words.size)
    H_noise = ...                                  # mean over time of the entropy of words across repeats
    return H_total, H_noise, H_total - H_noise

info, Htot = {}, {}
for name, jit in [("precise", 0.0), ("jittered", 3.0)]:
    Ht, Hn, I = direct_info(make_trains(jit), L)
    info[name], Htot[name] = I, Ht
    print(f"{name:9s}: total {Ht:.3f}, noise {Hn:.3f}, information {I:.3f} bits per 8 ms word ({I / 0.008:.0f} bits/s)")

In [ ]:
import numpy as np
def test_positive():
    "both neurons carry positive information, below their total entropy"
    assert all(0 < info[k] < Htot[k] for k in info)

def test_precise_more():
    "the precisely timed neuron carries at least 1.5 times as much information as the jittered one"
    assert info["precise"] > 1.5 * info["jittered"]

test_positive()
test_precise_more()
print("All checks passed")